In [12]:
import pickle
import pandas as pd

with open("/kaggle/input/datasets/shaambhavidubey/gnn-synthetic-data/gnn-upi/data/synthetic_graph.pkl", "rb") as f:
    DirGr = pickle.load(f)

node_df = pd.read_csv("/kaggle/input/datasets/shaambhavidubey/gnn-synthetic-data/gnn-upi/data/node_features.csv")
print(node_df.shape)
print(node_df["label"].value_counts())


(10000, 5)
label
0    9750
1     250
Name: count, dtype: int64


In [13]:
!git clone https://github.com/shaambhavi-dubey/gnn-upi.git

fatal: destination path 'gnn-upi' already exists and is not an empty directory.


In [14]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

github_token = secrets.get_secret("github_secret")   # note: matches your checked secret's name exactly

github_username = "shaambhavi-dubey"
repo_name = "gnn-upi"

new_remote = f"https://{github_token}@github.com/{github_username}/{repo_name}.git"
!cd gnn-upi && git remote set-url origin "{new_remote}"
!cd gnn-upi && git push

Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 4 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 532 bytes | 532.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/shaambhavi-dubey/gnn-upi.git
   79d7984..e2e9634  main -> main


In [15]:
!ls gnn-upi/data

node_features.csv	 results_06a_elliptic_tabular.json
results_02_tabular.json  synthetic_graph.pkl


In [16]:
# make tabular dataset without graph features (degree in/out) so we can rlly tell wso
amt_stat = []
for n in DirGr.nodes():
    incoming_m = [DirGr[u][n]['amount'] for u in DirGr.predecessors(n)]
    outgoing_m = [DirGr[n][t]['amount'] for t in DirGr.successors(n)]
    total = incoming_m + outgoing_m
    amt_stat.append({
        "node_id": n,
        "avg_amt": sum(total)/len(total) if total else 0,
        "max_amount": max(total) if total else 0,
    })

amount_df = pd.DataFrame(amt_stat)
tabular_df = node_df[["node_id", "account_age_days", "label"]].merge(amount_df, on="node_id")
print(tabular_df.shape)
print(tabular_df.head())

(10000, 5)
   node_id  account_age_days  label     avg_amt  max_amount
0        0               115      0   84.971419      527.30
1        1               490      0  107.677431     1049.31
2        2              1626      0   88.812000      350.70
3        3               633      0  139.980129     5559.58
4        4               720      0  112.883147     4161.46


In [17]:
from sklearn.model_selection import train_test_split
X = tabular_df.drop(columns=["node_id","label"])
y = tabular_df["label"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify = y,random_state=42)
print(f"Train fraud: {y_train.sum()}, Test fraud: {y_test.sum()}")

Train fraud: 200, Test fraud: 50


In [18]:
from xgboost import XGBClassifier

scale = (y_train == 0).sum() / (y_train == 1).sum()  # ratio of legit to fraud in train
print(f"scale_pos_weight: {scale:.1f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    scale_pos_weight=scale,
    eval_metric="aucpr",
    random_state=42
)
model.fit(X_train, y_train)

scale_pos_weight: 39.0


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [19]:
from sklearn.metrics import classification_report, average_precision_score, f1_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"PR-AUC: {average_precision_score(y_test, y_proba):.3f}")
print(f"F1: {f1_score(y_test, y_pred):.3f}")

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1950
           1       0.68      0.84      0.75        50

    accuracy                           0.99      2000
   macro avg       0.84      0.91      0.87      2000
weighted avg       0.99      0.99      0.99      2000

PR-AUC: 0.854
F1: 0.750


In [20]:
results = {
    "notebook": "02_baseline_xgboost_tabular",
    "features_used": ["account_age_days", "avg_amt", "max_amount"],
    "precision_fraud": 0.68,
    "recall_fraud": 0.84,
    "f1_fraud": 0.750,
    "pr_auc": 0.854
}

import json
with open("gnn-upi/data/results_02_tabular.json", "w") as f:
    json.dump(results, f, indent=2)

print("saved")

saved


In [21]:
!cd gnn-upi && git config user.email "25bit087@sot.pdpu.ac.in"
!cd gnn-upi && git config user.name "shaambhavi-dubey"

In [22]:
!cd gnn-upi && git add . && git commit -m "Notebook 02: XGBoost tabular-only baseline (F1=0.75, PR-AUC=0.854)"
!cd gnn-upi && git pull origin main --no-edit
!cd gnn-upi && git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
From https://github.com/shaambhavi-dubey/gnn-upi
 * branch            main       -> FETCH_HEAD
Already up to date.
Everything up-to-date
